# EEG Batch Converter
Converts all EEG files in a folder to CSV (20 min, 16 channels)

In [5]:
# ── Config ──────────────────────────────────────────────────────────────
INPUT_FOLDER  = r"C:\abalaji\bichat\ORIGINAL_DATA"
OUTPUT_FOLDER = r"C:\abalaji\bichat\ORIGINAL_DATA\csv_output"
SAMPLING_RATE = 256
MAX_SAMPLES   = 20 * 60 * SAMPLING_RATE  # 307,200 samples = 20 min

CHANNELS = ["Fp1","C3","P3","O1","F7","T3","T5","Cz","Pz",
            "Fp2","C4","P4","O2","F8","T4","T6"]
# ────────────────────────────────────────────────────────────────────────

In [7]:
import re
import csv

timestamp_line_re = re.compile(r'^\s*\d{2}:\d{2}:\d{2}\s+')
numeric_line_re   = re.compile(r'^\s*[-+]?\d+\.\d+')

def parse_eeg(input_path, output_path):
    sample_index = 0
    time_sec = 0.0
    dt = 1.0 / SAMPLING_RATE

    with open(input_path, "r", errors="ignore") as fin, \
         open(output_path, "w", newline="") as fout:

        writer = csv.writer(fout)
        writer.writerow(["Time"] + CHANNELS)

        for line in fin:
            if sample_index >= MAX_SAMPLES:
                break

            if timestamp_line_re.match(line):
                values = line.split()[1:]  # remove timestamp
            elif numeric_line_re.match(line):
                values = line.split()
            else:
                continue

            if len(values) < 16:
                continue

            eeg_vals = values[:16]

            writer.writerow(
                [f"{time_sec:.6f}"] + eeg_vals
            )

            time_sec += dt
            sample_index += 1

    return sample_index

In [8]:
# ── Run batch conversion ─────────────────────────────────────────────────
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

all_files = [
    f for f in os.listdir(INPUT_FOLDER)
    if os.path.isfile(os.path.join(INPUT_FOLDER, f)) and not f.startswith(".")
]

print(f"Found {len(all_files)} file(s)\n")

results = []

for fname in sorted(all_files):
    input_path  = os.path.join(INPUT_FOLDER, fname)
    stem        = os.path.splitext(fname)[0]
    output_path = os.path.join(OUTPUT_FOLDER, stem + ".csv")

    try:
        n    = parse_eeg(input_path, output_path)
        mins = round(n / SAMPLING_RATE / 60, 2)
        status = "✓"
        results.append((status, fname, n, mins, ""))
        print(f"  ✓ {fname:40s} → {n:>7,} samples ({mins} min)")
    except Exception as e:
        results.append(("✗", fname, 0, 0, str(e)))
        print(f"  ✗ {fname:40s} → ERROR: {e}")

print(f"\nDone. CSVs saved to:\n{OUTPUT_FOLDER}")

Found 97 file(s)

  ✓ 1214                                     → 307,200 samples (20.0 min)
  ✓ 5160                                     → 307,200 samples (20.0 min)
  ✓ 8058                                     → 307,200 samples (20.0 min)
  ✓ 8063                                     → 307,200 samples (20.0 min)
  ✓ 8064                                     → 307,200 samples (20.0 min)
  ✓ 8067                                     → 307,200 samples (20.0 min)
  ✓ 8093                                     → 307,200 samples (20.0 min)
  ✓ 8094                                     → 307,200 samples (20.0 min)
  ✓ 8097                                     → 307,200 samples (20.0 min)
  ✓ 8098                                     → 307,200 samples (20.0 min)
  ✓ 8099                                     → 307,200 samples (20.0 min)
  ✓ 8101                                     → 307,200 samples (20.0 min)
  ✓ 8102                                     → 307,200 samples (20.0 min)
  ✓ 8104            

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────
import pandas as pd

df_summary = pd.DataFrame(results, columns=["Status","File","Samples","Minutes","Error"])
display(df_summary)

In [ ]:
# ── Preview one CSV ──────────────────────────────────────────────────────
import pandas as pd

preview_file = results[0][1] if results else None

if preview_file:
    stem    = os.path.splitext(preview_file)[0]
    df      = pd.read_csv(os.path.join(OUTPUT_FOLDER, stem + ".csv"))
    print(f"Shape: {df.shape}")
    display(df.head(10))

In [1]:
import os
import shutil
import pandas as pd

excel_path = r"C:\abalaji\bichat\filtered.xlsx"
csv_dir = r"C:\abalaji\bichat\ORIGINAL_DATA\csv_output"

df = pd.read_excel(excel_path)

os.makedirs(os.path.join(csv_dir, "0"), exist_ok=True)
os.makedirs(os.path.join(csv_dir, "1"), exist_ok=True)

files = os.listdir(csv_dir)

for _, row in df.iterrows():
    n_pat = str(row["N_PAT"]).strip()

    try:
        mortality = int(row["Mortalité J28"])
    except:
        continue

    target_dir = os.path.join(csv_dir, str(mortality))

    for file in files:
        if not os.path.isfile(os.path.join(csv_dir, file)):
            continue

        filename_no_ext = os.path.splitext(file)[0]

        if filename_no_ext == n_pat:
            src = os.path.join(csv_dir, file)
            dst = os.path.join(target_dir, file)

            shutil.move(src, dst)
            print(f"Moved {file} -> {mortality}")
            break

Moved 8064.csv -> 1
Moved 8058.csv -> 0
Moved 8067.csv -> 0
Moved 8063.csv -> 0
Moved 8094.csv -> 1
Moved 8093.csv -> 1
Moved 8102.csv -> 0
Moved 8099.csv -> 0
Moved 8097.csv -> 0
Moved 8098.csv -> 0
Moved 8101.csv -> 0
Moved 8112.csv -> 0
Moved 8113.csv -> 0
Moved 8128.csv -> 1
Moved 8132.csv -> 0
Moved 8133.csv -> 0
Moved 8127.csv -> 0
Moved 8154.csv -> 1
Moved 8159.csv -> 0
Moved 8153.csv -> 0
Moved 8157.csv -> 0
Moved 8158.csv -> 0
Moved 8165.csv -> 0
Moved 8173.csv -> 0
Moved 8174.csv -> 1
Moved 8166.csv -> 0
Moved 8213.csv -> 0
Moved 8194.csv -> 1
Moved 8192.csv -> 1
Moved 8193.csv -> 0
Moved 8210.csv -> 0
Moved 8201.csv -> 0
Moved 8204.csv -> 1
Moved 8217.csv -> 1
Moved 8228.csv -> 1
Moved 8227.csv -> 1
Moved 8263.csv -> 0
Moved 8254.csv -> 0
Moved 8251.csv -> 0
Moved 8250.csv -> 1
Moved 8264.csv -> 0
Moved 8267.csv -> 0
Moved 8275.csv -> 1
Moved 8279.csv -> 1
Moved 8281.csv -> 0
Moved 8285.csv -> 0
Moved 8280.csv -> 1
Moved 8291.csv -> 0
Moved 8286.csv -> 1
Moved 8290.csv -> 1


In [27]:
import pandas as pd
import os

INPUT_BASE  = r"C:\abalaji\bichat\ORIGINAL_DATA\csv_output"
OUTPUT_BASE = r"C:\abalaji\bichat\ORIGINAL_DATA\chunks_20"
N = 20  # number of chunks per file
LABELS = [0, 1]


def equal_samples(df, n):
    """
    Cuts a DataFrame into n sub-DataFrames of equal length.
    n = number of resulting samples (NOT chunk size).
    Leftover rows (if any) are silently dropped.
    """
    step = int(len(df) / n)
    lower_limit = 0
    upper_limit = step
    samples = []
    while upper_limit <= len(df):
        samples.append(df[lower_limit:upper_limit])
        lower_limit = upper_limit
        upper_limit += step
    return samples


total_files   = 0
total_chunks  = 0

for label in LABELS:
    input_folder  = os.path.join(INPUT_BASE, str(label))
    output_folder = os.path.join(OUTPUT_BASE, str(label))
    os.makedirs(output_folder, exist_ok=True)

    csv_files = [f for f in os.listdir(input_folder) if f.endswith('.csv') or '.' not in f]

    print(f"\n── Folder: {input_folder}  (label={label}) ──")
    print(f"   Files found: {len(csv_files)}")

    for fname in csv_files:
        fpath = os.path.join(input_folder, fname)

        df = pd.read_csv(fpath)

        if 'Time' in df.columns:
            df = df.drop(columns=['Time'])

        df['label'] = label

        chunks = equal_samples(df, N)

        base_name = os.path.splitext(fname)[0]
        for i, chunk in enumerate(chunks):
            out_name = f"{base_name}_chunk_{i:03d}.csv"
            out_path = os.path.join(output_folder, out_name)
            chunk.to_csv(out_path, index=False)

        total_files  += 1
        total_chunks += len(chunks)

        print(f"   [{label}] {fname:20s} → shape {df.shape} → {len(chunks)} chunks "
              f"(each {chunks[0].shape[0]} rows)")

print(f"\n{'─'*50}")
print(f"Done.")
print(f"  Total files processed : {total_files}")
print(f"  Total chunks saved    : {total_chunks}")
print(f"  Output location       : {OUTPUT_BASE}")
print(f"{'─'*50}")


── Folder: C:\abalaji\bichat\ORIGINAL_DATA\csv_output\0  (label=0) ──
   Files found: 70
   [0] 1214.csv             → shape (307200, 17) → 20 chunks (each 15360 rows)
   [0] 8058.csv             → shape (307200, 17) → 20 chunks (each 15360 rows)
   [0] 8063.csv             → shape (307200, 17) → 20 chunks (each 15360 rows)
   [0] 8067.csv             → shape (307200, 17) → 20 chunks (each 15360 rows)
   [0] 8097.csv             → shape (307200, 17) → 20 chunks (each 15360 rows)
   [0] 8098.csv             → shape (307200, 17) → 20 chunks (each 15360 rows)
   [0] 8099.csv             → shape (307200, 17) → 20 chunks (each 15360 rows)
   [0] 8101.csv             → shape (307200, 17) → 20 chunks (each 15360 rows)
   [0] 8102.csv             → shape (307200, 17) → 20 chunks (each 15360 rows)
   [0] 8112.csv             → shape (307200, 17) → 20 chunks (each 15360 rows)
   [0] 8113.csv             → shape (307200, 17) → 20 chunks (each 15360 rows)
   [0] 8127.csv             → shape (3072

In [1]:
import pandas as pd

input_path = r"C:\abalaji\bichat\results\datasetTest.csv"

df = pd.read_csv(input_path)

df_8099 = df[df['label'] == 0].reset_index(drop=True)
df_8064 = df[df['label'] == 1].reset_index(drop=True)

df_8099.to_csv(r"C:\abalaji\bichat\results\datasetTest8099.csv", index=False)
df_8064.to_csv(r"C:\abalaji\bichat\results\datasetTest8064.csv", index=False)

print(f"datasetTest8099 (label=0): {len(df_8099)} rows")
print(f"datasetTest8064 (label=1): {len(df_8064)} rows")

datasetTest8099 (label=0): 20 rows
datasetTest8064 (label=1): 20 rows


In [2]:
input_path = r"C:\abalaji\bichat\results\datasetTrain.csv"

df = pd.read_csv(input_path)

print(f"datasetTrain {len(df)} rows")


datasetTrain 1860 rows


In [11]:
import pandas as pd

input_path = r"C:\abalaji\bichat\results\dataset\train99\1214_datasetSAPME_.csv"

df = pd.read_csv(input_path)

print(f"datasetTrain {len(df)} rows")


datasetTrain 15360 rows


In [21]:
import pandas as pd

input_path = r"C:\abalaji\bichat\ORIGINAL_DATA\1214_1dataset_.csv"

df = pd.read_csv(input_path)

print(f"datasetTrain {len(df)} rows")


datasetTrain 5120 rows


In [14]:
C:\abalaji\bichat\ORIGINAL_DATA\csv_output\0

SyntaxError: unexpected character after line continuation character (1061665901.py, line 1)

In [17]:
import pandas as pd

input_path = r"C:\abalaji\bichat\ORIGINAL_DATA\csv_output\0\1214.csv"

df = pd.read_csv(input_path)

print(f"datasetTrain {len(df)} rows")
print(f"datasetTrain shape {df.shape}")



datasetTrain 307200 rows
datasetTrain shape (307200, 17)


In [22]:
print(df.head(10))

          Time    Fp1     C3     P3     O1     F7     T3     T5    Cz     Pz  \
0  1180.000000   9.90   4.79   5.96  -8.88 -36.19 -35.22 -12.86  3.37 -19.41   
1  1180.003906   2.25   2.33  -3.76   3.04   4.26 -32.19  -0.36  7.49 -12.71   
2  1180.007812  12.82  10.06   3.80  -7.10 -35.57 -36.80 -14.31  8.53 -12.17   
3  1180.011719   5.06  -0.40  -5.58 -12.86 -17.91 -36.27 -11.92  3.60 -19.20   
4  1180.015625   6.81   5.88  -6.57  -3.53 -10.64 -38.01 -11.96  9.72  -8.65   
5  1180.019531   8.70   2.79  -2.45 -20.06 -45.00 -41.55 -24.62  2.09 -19.91   
6  1180.023438   1.51  -2.72 -10.83  -6.69  -1.86 -39.01 -10.56  3.55 -16.00   
7  1180.027344   9.72   6.28  -0.49  -7.52 -29.68 -40.73 -19.47  6.67 -11.61   
8  1180.031250   1.49  -3.67  -3.65 -13.58 -21.84 -37.65 -17.15 -0.82 -22.92   
9  1180.035156  -0.87   0.94  -2.05   2.67  -0.28 -33.40  -6.78  3.79 -13.75   

     Fp2    C4     P4     O2     F8     T4     T6  
0  -5.00 -0.29  -7.01 -18.61 -17.80 -23.10 -57.34  
1  -3.09 -2.86 

In [25]:
import pandas as pd

input_path = r"C:\abalaji\bichat\ORIGINAL_DATA\csv_output\0\1214.csv"

df = pd.read_csv(input_path)

def equal_samples(df, n):
    step = int(len(df) / n)
    lower_limit = 0
    upper_limit = step
    samples = []
    while upper_limit <= len(df):
        samples.append(df[lower_limit:upper_limit])
        lower_limit = upper_limit
        upper_limit += step
    return samples

n = 60
samples = equal_samples(df, n)

print(f"Input shape:     {df.shape}")
print(f"n (chunks):      {n}")
print(f"Step (rows/chunk): {int(len(df)/n)}")
print(f"Total chunks:    {len(samples)}")
print(f"Each chunk shape: {samples[0].shape}")

Input shape:     (307200, 17)
n (chunks):      60
Step (rows/chunk): 5120
Total chunks:    60
Each chunk shape: (5120, 17)


In [10]:
import warnings

info = mne.io.read_info('sub-1448.fif')
print(info)


<Info | 10 non-empty values
 bads: []
 ch_names: EEG001, EEG002, EEG003, EEG004, EEG005, EEG006, EEG007, EEG008, ...
 chs: 60 EEG, 2 EOG, 1 ECG
 custom_ref_applied: False
 dig: 63 items (3 Cardinal, 60 EEG)
 file_id: 4 items (dict)
 highpass: 0.5 Hz
 lowpass: 35.0 Hz
 meas_date: unspecified
 meas_id: 4 items (dict)
 nchan: 63
 projs: []
 sfreq: 1000.0 Hz
>


In [12]:
import matplotlib.pyplot as plt

raw = mne.io.read_raw_fif('sub-1448.fif', preload=True)  # or your actual file
raw.plot(duration=10, n_channels=20)
plt.show()

Opening raw data file sub-1448.fif...


C:\Users\abalaji\AppData\Local\Temp\ipykernel_7444\792004806.py:3: RuntimeWarning: This filename (sub-1448.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif('sub-1448.fif', preload=True)  # or your actual file


ValueError: No raw data in C:\abalaji\bichat\EEG_bichat\sub-1448.fif

In [13]:
from pathlib import Path
from mne._fiff.open import fiff_open

f, tree, _ = fiff_open(Path('sub-1448.fif'))
print([node.block for node in tree['children']])
f.close()

AttributeError: 'dict' object has no attribute 'block'